# Markov toy: future-return credit weights

Interactive port of `tests/special_standalone/future_returns_variance/`.

The student is a Markov policy $\pi(a\mid s)$ on $V$ states/actions with $s_{t+1}=a_t$.
The teacher is a fixed table. The exact gradient of

$$J=\sum_t \sum_s P_t(s)\, D_\alpha\big(\pi(\cdot\mid s),\,\mathrm{teacher}(\cdot\mid s)\big)$$

is available in closed form. Per-sample grads are autograd of the on-policy SDPO
full-logit surrogate

$$L_i=\sum_t \mathrm{mask}_t\big[D_\alpha(s_t)+W_t\log\pi(a_t\mid s_t)\big]$$

with stop-grad on $W_t$. Divergence $D_\alpha$ matches `compute_self_distillation_loss`:

| $\alpha$ | divergence |
| --- | --- |
| `0` | forward KL $\mathrm{KL}(\mathrm{teacher}\,\|\,\pi)$ |
| `1` | reverse KL $\mathrm{KL}(\pi\,\|\,\mathrm{teacher})$ |
| else | generalized JSD (default `0.5`) |

Helpers are imported from the standalone scripts so this notebook stays in sync
with the CLI. **Notebook defaults are smaller** than the CLI (`N=1024`, `T=64`)
so cells finish quickly. Bump the knobs, or skip the live runs and plot the
saved CLI results in the last section.


In [ ]:
from __future__ import annotations

import json
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F

CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / "verl").is_dir() else CWD.parent
TOY_DIR = ROOT / "tests" / "special_standalone" / "future_returns_variance"
SAVED = TOY_DIR / "results_onpolicy_fwdkl"

sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(TOY_DIR))

from variance_equal_vs_unequal import (  # noqa: E402
    VARIANTS as PADDED_VARIANTS,
    make_weights,
    metrics,
    per_sample_grads as per_sample_grads_masked,
    rollout as rollout_masked,
)
from variance_equal_vs_unequal import exact_grad as exact_grad_survival  # noqa: E402
from variance_batch_length_report import survival_uniform  # noqa: E402
from variance_study import (  # noqa: E402
    divergence_name,
    exact_grad,
    future_returns,
    per_sample_grads,
    remaining_weights,
    rollout as rollout_fixed,
)

torch.set_default_dtype(torch.float64)
plt.rcParams.update({"figure.figsize": (9.5, 4.6), "axes.grid": True, "grid.alpha": 0.3})
print("repo", ROOT)
print("toy helpers", TOY_DIR)


## Knobs

Live cells use these values. CLI-scale runs used `N=10000–20000`, `T∈{64,256,1024}`,
and 32–64 micro-batch repeats.


In [ ]:
V = 5
T = 64
N = 1024
ALPHA = 0.0  # 0=forward KL, 1=reverse KL, 0.5=JSD
SEED = 0
BATCH_SIZE = 16
REPS = 16
T_MIN_FRAC = 0.25
SWEEP_N_MIN, SWEEP_N_MAX = 2, 8  # N = 2**k

torch.manual_seed(SEED)
theta = torch.randn(V, V, requires_grad=True)
teacher_logits = torch.randn(V, V) * 1.5
teacher = F.softmax(teacher_logits, -1)
teacher_lp = F.log_softmax(teacher_logits, -1)
print(f"divergence={divergence_name(ALPHA)}  V={V}  T={T}  N={N}")

EQUAL_WEIGHT_FNS = {
    "raw G_t": lambda G, T, r: G,
    "G_t - batch_mean_t": lambda G, T, r: G - G.mean(0, keepdim=True),
    "G(r - batch_mean_t)": lambda G, T, r: future_returns(r - r.mean(0, keepdim=True)),
    "G_t / remaining": lambda G, T, r: G / remaining_weights(T),
    "(G_t - batch_mean_t)/remaining": lambda G, T, r: (G - G.mean(0, keepdim=True)) / remaining_weights(T),
    "G(r - batch_mean_t)/remaining": lambda G, T, r: future_returns(r - r.mean(0, keepdim=True)) / remaining_weights(T),
    "G_t / (T-1)": lambda G, T, r: G / (T - 1),
    "G(r - batch_mean_t)/(T-1)": lambda G, T, r: future_returns(r - r.mean(0, keepdim=True)) / (T - 1),
    "no PG term": lambda G, T, r: torch.zeros_like(G),
}


def grad_metrics(g: torch.Tensor, g_star: torch.Tensor) -> dict:
    mean = g.mean(0)
    noise = (g - mean).pow(2).sum(dim=(1, 2)).mean().sqrt()
    snr = (mean.norm() / noise).item() if noise > 0 else float("nan")
    cos = F.cosine_similarity(mean.flatten(), g_star.flatten(), dim=0).item()
    return {"snr": snr, "cos": cos, "mean_norm": mean.norm().item(), "noise": noise.item()}


def plot_snr_cos(df: pd.DataFrame, title: str, variant_col: str = "variant"):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
    order = list(df[variant_col])
    axes[0].barh(order, df["snr"])
    axes[0].set_xlabel("SNR")
    axes[0].invert_yaxis()
    axes[1].barh(order, df["cos"])
    axes[1].set_xlabel("cosine to exact grad")
    axes[1].set_xlim(min(0.9, df["cos"].min() - 0.02), 1.002)
    axes[1].invert_yaxis()
    fig.suptitle(title)
    fig.tight_layout()
    return fig


## 1. Credit-weight comparison (equal lengths)

Port of `variance_study.py`. Compares estimators of $W_t$ on a fixed-length
on-policy pool. Unbiased rules (`G_t - \mathrm{mean}_t(G_t)` and
`G(r-\mathrm{mean}_t(r))`) should sit at cosine ≈ 1; `/remaining` (DISTIL) and
`no PG` trade bias for lower variance.


In [ ]:
g_star = exact_grad(theta, teacher, teacher_lp, T, alpha=ALPHA)
st, ac, rw = rollout_fixed(theta, teacher, teacher_lp, T, N, alpha=ALPHA)
print(f"||exact||={g_star.norm():.3f}  mean d_t={rw.mean():.4f}")

rows = []
for name, fn in EQUAL_WEIGHT_FNS.items():
    g = per_sample_grads(theta, teacher, st, ac, rw, fn, teacher_lp=teacher_lp, alpha=ALPHA)
    rows.append({"variant": name, **grad_metrics(g, g_star)})

credit_df = pd.DataFrame(rows)
display(credit_df.round(5))
plot_snr_cos(credit_df, f"Equal-length credit weights  {divergence_name(ALPHA)}  T={T}  N={N}")
plt.show()


## 2. Equal vs unequal lengths

Port of `variance_equal_vs_unequal.py`. Unequal trajectories are right-padded
with $L\sim\mathrm{Uniform}\{\lceil f T\rceil,\ldots,T\}$. The exact gradient
multiplies each timestep by survival $P(L>t)$. Micro-batch rows recompute the
position-wise baseline inside each subset of size `BATCH_SIZE`.


In [ ]:
T_min = max(2, int(math.ceil(T_MIN_FRAC * T)))
lengths = torch.randint(T_min, T + 1, (N,))
survival = survival_uniform(T, T_min)
print(f"unequal T_min={T_min}  E[L]={(T_min + T) / 2:.1f}  survival[0]={survival[0]:.3f}")


def eval_setting(label, lengths=None, survival=None):
    g_star = exact_grad_survival(theta, teacher, teacher_lp, T, survival=survival, alpha=ALPHA)
    st, ac, rw, mask = rollout_masked(
        theta, teacher, teacher_lp, T, N, lengths=lengths, alpha=ALPHA
    )
    print(
        f"{label}: ||exact||={g_star.norm():.3f}  mean_len={mask.sum(1).mean():.1f}  "
        f"mean_r={rw.mean():.4f}"
    )
    rows = []
    for name in PADDED_VARIANTS:
        W = make_weights(name, rw, mask, T)
        g = per_sample_grads_masked(
            theta, teacher, st, ac, rw, mask, W, teacher_lp=teacher_lp, alpha=ALPHA
        )
        rows.append({"setting": label, "scope": "full_pool", "variant": name, **metrics(g, g_star)})

    acc = {name: {"cos": [], "snr": []} for name in PADDED_VARIANTS}
    for _ in range(REPS):
        idx = torch.randperm(N)[:BATCH_SIZE]
        st_b, ac_b, rw_b, mask_b = st[idx], ac[idx], rw[idx], mask[idx]
        for name in PADDED_VARIANTS:
            W = make_weights(name, rw_b, mask_b, T)
            g = per_sample_grads_masked(
                theta, teacher, st_b, ac_b, rw_b, mask_b, W, teacher_lp=teacher_lp, alpha=ALPHA
            )
            m = metrics(g, g_star)
            acc[name]["cos"].append(m["cos"])
            acc[name]["snr"].append(m["snr"])
    for name in PADDED_VARIANTS:
        cos = torch.tensor(acc[name]["cos"])
        snr = torch.tensor(acc[name]["snr"])
        rows.append({
            "setting": label,
            "scope": "micro_batch",
            "variant": name,
            "cos": cos.mean().item(),
            "cos_std": cos.std().item(),
            "snr": snr.mean().item(),
            "snr_std": snr.std().item(),
        })
    return pd.DataFrame(rows)


eq_df = eval_setting("equal_lengths")
uq_df = eval_setting("unequal_lengths", lengths=lengths, survival=survival)
len_df = pd.concat([eq_df, uq_df], ignore_index=True)

micro = len_df[len_df.scope == "micro_batch"].pivot(index="variant", columns="setting", values=["cos", "snr"])
micro.columns = [f"{a}_{b}" for a, b in micro.columns]
micro = micro.reindex(PADDED_VARIANTS)
micro["abs_d_cos"] = (micro["cos_equal_lengths"] - micro["cos_unequal_lengths"]).abs()
display(micro.round(5))

fig, ax = plt.subplots(figsize=(10, 4.8))
x = range(len(PADDED_VARIANTS))
ax.bar([i - 0.2 for i in x], micro["cos_equal_lengths"], 0.4, label="equal")
ax.bar([i + 0.2 for i in x], micro["cos_unequal_lengths"], 0.4, label="unequal")
ax.set_xticks(list(x), PADDED_VARIANTS, rotation=35, ha="right")
ax.set_ylabel("micro-batch cosine to exact")
ax.set_title(f"Equal vs unequal  B={BATCH_SIZE}  {REPS} subsets  {divergence_name(ALPHA)}")
ax.legend()
fig.tight_layout()
plt.show()


## 3. Batch-size sweep (equal lengths)

Port of `variance_batch_size_sweep.py`. For each $N=2^k$, the position-wise
baseline is computed on that same subset (as in a training step). Metrics are
averaged over `REPS` random draws from a fixed rollout pool.


In [ ]:
NS = [2 ** k for k in range(SWEEP_N_MIN, SWEEP_N_MAX + 1)]
N_POOL = max(NS[-1], N)
g_star = exact_grad(theta, teacher, teacher_lp, T, alpha=ALPHA)
st_all, ac_all, rw_all = rollout_fixed(theta, teacher, teacher_lp, T, N_POOL, alpha=ALPHA)
print(f"pool N={N_POOL}  ||exact||={g_star.norm():.3f}")

sweep_rows = []
for n in NS:
    print(f"N={n}", flush=True)
    acc = {name: {"snr": [], "cos": []} for name in EQUAL_WEIGHT_FNS}
    for _ in range(REPS):
        idx = torch.randperm(N_POOL)[:n]
        st, ac, rw = st_all[idx], ac_all[idx], rw_all[idx]
        for name, fn in EQUAL_WEIGHT_FNS.items():
            g = per_sample_grads(theta, teacher, st, ac, rw, fn, teacher_lp=teacher_lp, alpha=ALPHA)
            m = grad_metrics(g, g_star)
            acc[name]["snr"].append(m["snr"])
            acc[name]["cos"].append(m["cos"])
    for name in EQUAL_WEIGHT_FNS:
        snrs = torch.tensor([x for x in acc[name]["snr"] if not math.isnan(x)])
        coss = torch.tensor(acc[name]["cos"])
        sweep_rows.append({
            "N": n,
            "variant": name,
            "snr_mean": snrs.mean().item() if len(snrs) else float("nan"),
            "snr_std": snrs.std(unbiased=False).item() if len(snrs) else float("nan"),
            "cos_mean": coss.mean().item(),
            "cos_std": coss.std(unbiased=False).item(),
        })

sweep_df = pd.DataFrame(sweep_rows)
display(sweep_df.pivot(index="N", columns="variant", values="cos_mean").round(4))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
for name, sub in sweep_df.groupby("variant"):
    axes[0].plot(sub["N"], sub["cos_mean"], marker="o", label=name)
    axes[1].plot(sub["N"], sub["snr_mean"], marker="o", label=name)
axes[0].set_xscale("log", base=2)
axes[1].set_xscale("log", base=2)
axes[0].set_xlabel("batch size N")
axes[1].set_xlabel("batch size N")
axes[0].set_ylabel("cosine to exact")
axes[1].set_ylabel("SNR")
axes[0].set_title(f"Cosine vs N  T={T}  {divergence_name(ALPHA)}")
axes[1].set_title("SNR vs N")
axes[0].legend(fontsize=8, loc="lower right")
fig.tight_layout()
plt.show()


## 4. Batch size × length regime

Port of `variance_batch_length_report.py`. Sweeps micro-batch size under both
equal and unequal lengths on one shared pool.


In [ ]:
BS = [2 ** k for k in range(SWEEP_N_MIN, min(SWEEP_N_MAX, int(math.log2(N))) + 1)]
T_min = max(2, int(math.ceil(T_MIN_FRAC * T)))
settings = {
    "equal_lengths": {"lengths": None, "survival": None},
    "unequal_lengths": {
        "lengths": torch.randint(T_min, T + 1, (N,)),
        "survival": survival_uniform(T, T_min),
    },
}

len_sweep_rows = []
for setting, cfg in settings.items():
    g_star = exact_grad_survival(theta, teacher, teacher_lp, T, survival=cfg["survival"], alpha=ALPHA)
    st, ac, rw, mask = rollout_masked(
        theta, teacher, teacher_lp, T, N, lengths=cfg["lengths"], alpha=ALPHA
    )
    print(f"{setting}: ||exact||={g_star.norm():.3f}  mean_len={mask.sum(1).mean():.1f}", flush=True)
    for B in BS:
        acc = {name: {"cos": [], "snr": []} for name in PADDED_VARIANTS}
        for _ in range(REPS):
            idx = torch.randperm(N)[:B]
            st_b, ac_b, rw_b, mask_b = st[idx], ac[idx], rw[idx], mask[idx]
            for name in PADDED_VARIANTS:
                W = make_weights(name, rw_b, mask_b, T)
                g = per_sample_grads_masked(
                    theta, teacher, st_b, ac_b, rw_b, mask_b, W, teacher_lp=teacher_lp, alpha=ALPHA
                )
                m = metrics(g, g_star)
                acc[name]["cos"].append(m["cos"])
                acc[name]["snr"].append(m["snr"])
        for name in PADDED_VARIANTS:
            cos = torch.tensor(acc[name]["cos"])
            snr = torch.tensor(acc[name]["snr"])
            len_sweep_rows.append({
                "setting": setting,
                "B": B,
                "variant": name,
                "cos_mean": cos.mean().item(),
                "cos_std": cos.std(unbiased=False).item(),
                "snr_mean": snr.mean().item(),
                "snr_std": snr.std(unbiased=False).item(),
            })

len_sweep_df = pd.DataFrame(len_sweep_rows)
for setting in settings:
    print(f"\n{setting}")
    display(
        len_sweep_df[len_sweep_df.setting == setting]
        .pivot(index="B", columns="variant", values="cos_mean")
        .round(4)
    )

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), sharey=True)
for ax, setting in zip(axes, settings):
    sub = len_sweep_df[len_sweep_df.setting == setting]
    for name, grp in sub.groupby("variant"):
        ax.plot(grp["B"], grp["cos_mean"], marker="o", label=name)
    ax.set_xscale("log", base=2)
    ax.set_xlabel("micro-batch B")
    ax.set_title(setting)
axes[0].set_ylabel("cosine to exact")
axes[1].legend(fontsize=7, loc="lower right")
fig.suptitle(f"Cosine vs B  T={T}  {divergence_name(ALPHA)}")
fig.tight_layout()
plt.show()


## 5. Saved CLI results

The standalone scripts already wrote CLI-scale JSON under
`tests/special_standalone/future_returns_variance/results_onpolicy_fwdkl/`.
This cell plots those without re-running.


In [ ]:
SAVED_BY_ALPHA = {
    0.0: {
        "sweep": "batch_size_sweep_forward_kl.json",
        "equal": "equal_vs_unequal_forward_kl.json",
        "length": "batch_length_forward_kl.json",
    },
    1.0: {
        "sweep": "batch_size_sweep_reverse_kl.json",
        "equal": "equal_vs_unequal_reverse_kl.json",
        "length": "batch_length_reverse_kl.json",
    },
    0.5: {
        "sweep": "batch_size_sweep_jsd.json",
        "equal": "equal_vs_unequal_jsd.json",
        "length": "batch_length_jsd.json",
    },
}


def load_json(name: str):
    path = SAVED / name
    with path.open() as f:
        return json.load(f), path


files = SAVED_BY_ALPHA.get(ALPHA, SAVED_BY_ALPHA[0.0])
sweep, sweep_path = load_json(files["sweep"])
equal_rows, equal_path = load_json(files["equal"])
length, length_path = load_json(files["length"])
print("loaded")
print(" ", sweep_path.name)
print(" ", equal_path.name)
print(" ", length_path.name)

# Batch-size sweep
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
for name, payload in sweep["results"].items():
    axes[0].plot(payload["N"], payload["cos_mean"], marker="o", label=name)
    axes[1].plot(payload["N"], payload["snr_mean"], marker="o", label=name)
axes[0].set_xscale("log", base=2)
axes[1].set_xscale("log", base=2)
axes[0].set_xlabel("batch size N")
axes[1].set_xlabel("batch size N")
axes[0].set_ylabel("cosine to exact")
axes[1].set_ylabel("SNR")
alpha = sweep.get("alpha", ALPHA)
axes[0].set_title(f"Saved cosine vs N  T={sweep['T']}  {divergence_name(alpha)}")
axes[1].set_title("Saved SNR vs N")
axes[0].legend(fontsize=8, loc="lower right")
fig.tight_layout()
plt.show()

# Equal vs unequal, micro-batch cosine
eq = pd.DataFrame(equal_rows)
micro = eq[eq.scope == "micro_batch"]
fig, ax = plt.subplots(figsize=(10, 4.8))
variants = list(dict.fromkeys(micro["variant"]))
Ts = sorted(micro["T"].unique())
width = 0.35
for i, t in enumerate(Ts):
    sub = micro[micro["T"] == t]
    eq_cos = [sub[(sub.variant == v) & (sub.setting == "equal_lengths")]["cos"].iloc[0] for v in variants]
    uq_cos = [sub[(sub.variant == v) & (sub.setting == "unequal_lengths")]["cos"].iloc[0] for v in variants]
    x = [j + (i - 0.5) * width for j in range(len(variants))]
    ax.bar(x, eq_cos, width / 2, label=f"T={t} equal")
    ax.bar([xi + width / 2 for xi in x], uq_cos, width / 2, label=f"T={t} unequal")
ax.set_xticks(range(len(variants)), variants, rotation=35, ha="right")
ax.set_ylabel("micro-batch cosine to exact")
ax.set_title(f"Saved equal vs unequal  {equal_path.stem}")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

# Batch × length
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), sharey=True)
meta = length["meta"]
for ax, setting in zip(axes, ("equal_lengths", "unequal_lengths")):
    by_B = length["settings"][setting]["by_B"]
    Bs = [int(b) for b in by_B]
    for name in meta["variants"]:
        ax.plot(Bs, [by_B[str(b)][name]["cos_mean"] for b in Bs], marker="o", label=name)
    ax.set_xscale("log", base=2)
    ax.set_xlabel("micro-batch B")
    ax.set_title(setting)
axes[0].set_ylabel("cosine to exact")
axes[1].legend(fontsize=7, loc="lower right")
fig.suptitle(
    f"Saved cosine vs B  T={meta['T']}  alpha={meta.get('alpha', ALPHA)}  {length_path.stem}"
)
fig.tight_layout()
plt.show()
